
# Bone-marrow HAI benchmark

This notebook evaluates:

1. GraphHDBSCAN*
2. HDBSCAN*
3. HDBSCAN* + PCA
4. AHSCAN
5. DHSCAN
6. Paris

For every graph-based method, the initial similarity graph is created with the
GraphHDBSCAN* package using `k=15`, Euclidean distance, and each of:

- `sc_umap`
- `sc_gauss`
- `jaccard_phenograph`



In [ ]:

%pip install -q --upgrade pip
%pip install -q \
    "setuptools<82" \
    "Cython>=3,<4" \
    wheel numpy pandas scipy scikit-learn \
    anndata scanpy hdbscan umap-learn phenograph numba networkx

%pip uninstall -y scikit-network
%pip install -q \
    --no-cache-dir \
    --no-build-isolation \
    --no-binary=scikit-network \
    --no-deps \
    "scikit-network==0.33.5"

%pip install -q --no-cache-dir \
    git+https://github.com/Campello-Lab/GraphHDBSCAN.git


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Found existing installation: scikit-network 0.33.5
Uninstalling scikit-network-0.33.5:
  Successfully uninstalled scikit-network-0.33.5


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


## 1. Configuration

In [ ]:

from pathlib import Path
from collections import defaultdict
import gc
import heapq
import math
import time
import zipfile

import numpy as np
import pandas as pd
import anndata as ad
import networkx as nx
import hdbscan

from scipy import sparse
from sklearn.decomposition import PCA
from IPython.display import display

from coresg_graphhdbscan import GraphCoreSGHDBSCAN
from sknetwork.hierarchy import Paris

LEVEL_KEYS = ["Level 1", "Level 2", "Level 3 Multimodal"]

K_NEIGHBORS = 15
GRAPH_METRIC = "euclidean"
GRAPH_METHODS = ["sc_umap", "sc_gauss", "jaccard_phenograph"]

MIN_CLUSTER_SIZE = 55
MIN_SAMPLES_VALUES = list(range(2, 51))

ADD_NEIGHBOR = True
PCA_COMPONENTS = 15

RUN_AHSCAN = True
RUN_DHSCAN = True
RUN_PARIS = True

COLLAPSE_SAME_NAME_L3_TO_PARENT = True

EXPLICIT_DATA_PATH = None


## 2. Load dataset

In [ ]:

def locate_dataset():
    if EXPLICIT_DATA_PATH is not None:
        p = Path(EXPLICIT_DATA_PATH).expanduser().resolve()
        if not p.exists():
            raise FileNotFoundError(p)
        return p

    cwd = Path.cwd()

    direct = [
        cwd / "bone_marrow_processed.h5ad",
        cwd / "new dataset" / "bone_marrow_processed.h5ad",
        Path("/content/bone_marrow_processed.h5ad"),
        Path("/mnt/data/bone_marrow_processed.h5ad"),
    ]

    for p in direct:
        if p.exists():
            return p.resolve()

    zips = [
        cwd / "new dataset.zip",
        Path("/content/new dataset.zip"),
        Path("/mnt/data/new dataset.zip"),
    ]

    zip_path = next((p for p in zips if p.exists()), None)

    if zip_path is None:
        raise FileNotFoundError(
            "Upload 'new dataset.zip' or 'bone_marrow_processed.h5ad'."
        )

    extract_dir = cwd / "bone_marrow_dataset"
    extract_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as zf:
        members = [
            name for name in zf.namelist()
            if name.endswith("bone_marrow_processed.h5ad")
            and "__MACOSX" not in name
        ]

        if len(members) != 1:
            raise ValueError(
                "Expected exactly one bone_marrow_processed.h5ad; "
                f"found {members}"
            )

        zf.extract(members[0], extract_dir)
        return (extract_dir / members[0]).resolve()


DATA_PATH = locate_dataset()
adata = ad.read_h5ad(DATA_PATH)

missing = [key for key in LEVEL_KEYS if key not in adata.obs.columns]
if missing:
    raise KeyError(f"Missing hierarchy columns: {missing}")

X = adata.X
N = adata.n_obs

X_dense = X.toarray() if sparse.issparse(X) else np.asarray(X)
X_dense = np.asarray(X_dense, dtype=np.float64, order="C")

print("Dataset:", DATA_PATH)
print(adata)
print("Cells:", N)
print("Features:", adata.n_vars)


Dataset: /work/bone_marrow_processed.h5ad
AnnData object with n_obs × n_vars = 71260 × 1000
    obs: 'sample', 'ancestry', 'sex', 'age', 'capture', 'Level 1', 'Level 2', 'Level 3 Multimodal', 'Level 3 RNA', 'confidence', 'tfidf1', 'Donor', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'log1p'
    layers: 'counts', None (.X)
Cells: 71260
Features: 1000


## 3. Biological reference

In [ ]:

def prepare_reference_hierarchy(
    adata,
    level_keys=LEVEL_KEYS,
    collapse_same_name_l3=True,
):
    """
    Build a reference hierarchy that allows cells to attach directly
    to an internal Level-2 node.
    """
    df = adata.obs[level_keys].copy()

    if df.isna().any().any():
        bad = df.columns[df.isna().any()].tolist()
        raise ValueError(f"Missing hierarchy labels in {bad}")

    for col in level_keys:
        df[col] = df[col].astype(str)

    n = len(df)
    ROOT = ("ROOT",)

    sizes = defaultdict(int)
    children = defaultdict(set)
    attachments = []
    direct_parent_counts = defaultdict(int)

    for l1, l2, l3 in df.itertuples(index=False, name=None):
        node1 = ("L1", l1)
        node2 = ("L2", l1, l2)
        node3 = ("L3", l1, l2, l3)

        sizes[ROOT] += 1
        sizes[node1] += 1
        sizes[node2] += 1

        children[ROOT].add(node1)
        children[node1].add(node2)

        if collapse_same_name_l3 and l3 == l2:
            attachment = node2
            direct_parent_counts[node2] += 1
        else:
            attachment = node3
            sizes[node3] += 1
            children[node2].add(node3)

        attachments.append(attachment)

    unique_nodes = sorted(
        set(attachments),
        key=lambda node: tuple(map(str, node)),
    )

    code_of = {node: i for i, node in enumerate(unique_nodes)}

    encoded = np.fromiter(
        (code_of[node] for node in attachments),
        dtype=np.int32,
        count=n,
    )

    def ancestors(node):
        kind = node[0]

        if kind == "L3":
            _, l1, l2, _ = node
            return [node, ("L2", l1, l2), ("L1", l1), ROOT]

        if kind == "L2":
            _, l1, _ = node
            return [node, ("L1", l1), ROOT]

        if kind == "L1":
            return [node, ROOT]

        if kind == "ROOT":
            return [ROOT]

        raise ValueError(f"Unknown node: {node}")

    q = len(unique_nodes)
    gt_distance = np.zeros((q, q), dtype=np.float64)

    for i, node_i in enumerate(unique_nodes):
        anc_i = ancestors(node_i)

        for j, node_j in enumerate(unique_nodes):
            anc_j = set(ancestors(node_j))
            lca = next(node for node in anc_i if node in anc_j)

            # Neto/Campello HAI:
            # distance = size of the smallest common cluster / N.
            # Terminal biological clusters are still clusters, so distinct
            # cells meeting there use size(lca)/N rather than 0.
            distance = sizes[lca] / n

            gt_distance[i, j] = distance

    direct_rows = []

    for node, count in sorted(
        direct_parent_counts.items(),
        key=lambda item: tuple(map(str, item[0])),
    ):
        direct_rows.append(
            {
                "reference_node": " > ".join(map(str, node[1:])),
                "directly_attached_cells": int(count),
                "node_size": int(sizes[node]),
                "has_child_nodes": bool(children.get(node)),
                "HAI_same_attachment_distance": sizes[node] / n,
            }
        )

    return {
        "encoded": encoded,
        "distance": gt_distance,
        "label_order": unique_nodes,
        "sizes": dict(sizes),
        "children": {key: set(value) for key, value in children.items()},
        "direct_assignments": pd.DataFrame(direct_rows),
    }


GT = prepare_reference_hierarchy(
    adata,
    collapse_same_name_l3=COLLAPSE_SAME_NAME_L3_TO_PARENT,
)

print("Reference attachment groups:", len(GT["label_order"]))
print("Repeated-name Level-3 assignments attached to Level 2:")
display(GT["direct_assignments"])


Reference attachment groups: 89
Repeated-name Level-3 assignments attached to Level 2:


,reference_node,directly_attached_cells,node_size,has_child_nodes,HAI_same_attachment_distance
0,B cell > Plasma Cell,87,87,False,0.001221
1,Dendritic > ASDC,583,583,False,0.008181
2,Dendritic > Mac,46,46,False,0.000646
3,Dendritic > pDC,2395,2395,False,0.033609
4,Early-lymphoid > CLP,1785,6346,True,0.089054
5,Early-lymphoid > pre-B,472,4069,True,0.057101


## 4. Exact HAI utilities

In [ ]:

def condensed_tree_hai(condensed_tree, gt=GT):
    """
    Exact HAI for an HDBSCAN-style condensed tree.
    """
    n = len(gt["encoded"])
    encoded = gt["encoded"]
    gt_distance = gt["distance"]
    n_labels = len(gt["label_order"])

    tree = (
        condensed_tree.to_pandas()
        if hasattr(condensed_tree, "to_pandas")
        else pd.DataFrame(condensed_tree)
    )

    tree = tree[["parent", "child", "child_size"]].copy()
    tree["parent"] = tree["parent"].astype(np.int64)
    tree["child"] = tree["child"].astype(np.int64)

    parents = set(tree["parent"])
    child_nodes = set(tree["child"])
    roots = sorted(parents - child_nodes)

    if len(roots) != 1:
        raise ValueError(f"Expected one condensed-tree root; found {roots}")

    root = int(roots[0])
    children = defaultdict(list)
    observed_points = set()

    for parent, child in tree[["parent", "child"]].itertuples(index=False):
        parent = int(parent)
        child = int(child)
        children[parent].append(child)

        if child < n:
            observed_points.add(child)

    missing = set(range(n)) - observed_points

    if missing:
        raise ValueError(
            f"{len(missing)} observations are absent from the condensed tree."
        )

    postorder = []
    stack = [(root, False)]

    while stack:
        node, expanded = stack.pop()

        if expanded:
            postorder.append(node)
        else:
            stack.append((node, True))
            for child in children.get(node, []):
                if child >= n:
                    stack.append((child, False))

    histograms = {}
    sizes = {}
    absolute_difference = 0.0
    counted_pairs = 0

    for node in postorder:
        node_children = children.get(node, [])

        histogram = np.zeros(n_labels, dtype=np.int64)
        within_children = np.zeros((n_labels, n_labels), dtype=np.int64)
        for child in node_children:
            if child < n:
                label = int(encoded[child])
                histogram[label] += 1
                within_children[label, label] += 1
            else:
                child_hist = histograms[child]
                histogram += child_hist
                within_children += np.outer(child_hist, child_hist)

        size = int(histogram.sum())

        histograms[node] = histogram
        sizes[node] = size

        pair_counts = np.outer(histogram, histogram) - within_children

        # Neto/Campello HAI: a terminal condensed cluster is still the
        # smallest common cluster for the distinct point pairs that meet here.
        predicted_distance = size / n

        absolute_difference += np.sum(
            pair_counts * np.abs(predicted_distance - gt_distance),
            dtype=np.float64,
        )

        counted_pairs += int(pair_counts.sum())

    if sizes[root] != n:
        raise ValueError(
            f"Condensed-tree root has {sizes[root]} cells; expected {n}."
        )

    expected_pairs = n * (n - 1)

    if counted_pairs != expected_pairs:
        raise ValueError(
            f"Counted {counted_pairs} ordered pairs; expected {expected_pairs}."
        )

    return float(1.0 - absolute_difference / (n * n))


def reference_histogram(nodes, gt=GT):
    nodes = np.asarray(list(nodes), dtype=np.int64)

    return np.bincount(
        gt["encoded"][nodes],
        minlength=len(gt["label_order"]),
    ).astype(np.int64)


def forest_root_contribution(components, gt=GT):
    """
    Error contribution for a synthetic root above disconnected components.
    Every cross-component pair has inferred hierarchy distance 1.
    """
    n_labels = len(gt["label_order"])

    histograms = [
        reference_histogram(component, gt=gt)
        for component in components
    ]

    total = np.zeros(n_labels, dtype=np.int64)
    within = np.zeros((n_labels, n_labels), dtype=np.int64)

    for hist in histograms:
        total += hist
        within += np.outer(hist, hist)

    cross_pairs = np.outer(total, total) - within

    error = np.sum(
        cross_pairs * np.abs(1.0 - gt["distance"]),
        dtype=np.float64,
    )

    return float(error), int(cross_pairs.sum())


In [ ]:
# ============================================================
# Neto/Campello HAI audit
# ============================================================
# For DISTINCT cells whose reference attachment node is the same:
# d_GT = size(reference node) / N.
#
# Self-pairs are omitted from the compressed numerator because their
# HAI error is exactly zero; the final denominator remains N^2.

_n = len(GT["encoded"])
_expected_diag = np.array(
    [GT["sizes"][node] / _n for node in GT["label_order"]],
    dtype=np.float64,
)

assert np.allclose(
    np.diag(GT["distance"]),
    _expected_diag,
), "Reference HAI same-attachment distance must equal cluster_size / N."

assert np.all((GT["distance"] >= 0.0) & (GT["distance"] <= 1.0))
assert np.isclose(np.max(GT["distance"]), 1.0)

print("Neto/Campello HAI reference audit: PASSED")
print("N =", _n)


Neto/Campello HAI reference audit: PASSED
N = 71260


## 5. Build raw similarity graphs with GraphHDBSCAN*

In [ ]:

SIMILARITY_GRAPHS = {}
GRAPH_BUILD_INFO = []

for graph_method in GRAPH_METHODS:
    print(
        f"\nBuilding {graph_method} "
        f"(k={K_NEIGHBORS}, metric={GRAPH_METRIC})"
    )

    builder = GraphCoreSGHDBSCAN(
        min_samples=2,
        sim_graph_method=graph_method,
        metric=GRAPH_METRIC,
        n_neighbors=K_NEIGHBORS,
        min_cluster_size=MIN_CLUSTER_SIZE,
        add_neighbor=ADD_NEIGHBOR,
        no_noise=False,
        similarity_backend="auto",
        use_sparse_fit=True,
    )

    start = time.perf_counter()
    graph = builder.create_similarity_graph(X)
    runtime = time.perf_counter() - start

    graph = nx.Graph(graph)
    graph.add_nodes_from(range(N))
    graph.remove_edges_from(nx.selfloop_edges(graph))

    if set(graph.nodes()) != set(range(N)):
        raise ValueError(
            f"{graph_method}: nodes are not exactly 0,...,{N - 1}."
        )

    components = nx.number_connected_components(graph)

    SIMILARITY_GRAPHS[graph_method] = graph

    weights = np.fromiter(
        (
            float(data.get("weight", 1.0))
            for _, _, data in graph.edges(data=True)
        ),
        dtype=np.float64,
    )

    GRAPH_BUILD_INFO.append(
        {
            "graph_method": graph_method,
            "metric": GRAPH_METRIC,
            "k": K_NEIGHBORS,
            "nodes": graph.number_of_nodes(),
            "edges": graph.number_of_edges(),
            "connected_components": components,
            "min_weight": float(weights.min()) if weights.size else np.nan,
            "max_weight": float(weights.max()) if weights.size else np.nan,
            "construction_runtime_seconds": runtime,
        }
    )

    # Release the package's retained dense distance workspace before
    # constructing the next graph.
    del builder
    gc.collect()


graph_build_info = pd.DataFrame(GRAPH_BUILD_INFO)

display(
    graph_build_info.style.format(
        {
            "min_weight": "{:.6g}",
            "max_weight": "{:.6g}",
            "construction_runtime_seconds": "{:.2f}",
        }
    )
)



Building sc_umap (k=15, metric=euclidean)


[TIMER] GraphCoreSGHDBSCAN._create_similarity_sparse: 214.2727s


[TIMER] GraphCoreSGHDBSCAN.create_similarity_graph: 218.3005s



Building sc_gauss (k=15, metric=euclidean)


[TIMER] GraphCoreSGHDBSCAN._create_similarity_sparse: 45.6131s


[TIMER] GraphCoreSGHDBSCAN.create_similarity_graph: 49.9185s



Building jaccard_phenograph (k=15, metric=euclidean)


[TIMER] GraphCoreSGHDBSCAN._fast_phenograph_jaccard_from_knn_graph: 1.1378s
[TIMER] GraphCoreSGHDBSCAN._create_similarity_sparse: 197.3683s


[TIMER] GraphCoreSGHDBSCAN.create_similarity_graph: 198.7565s


,graph_method,metric,k,nodes,edges,connected_components,min_weight,max_weight,construction_runtime_seconds
0,sc_umap,euclidean,15,71260,927166,1,3.62029e-10,1,218.30
1,sc_gauss,euclidean,15,71260,927166,1,0.341323,0.870048,49.92
2,jaccard_phenograph,euclidean,15,71260,799280,2,0.0185185,0.866667,198.76


## 6. GraphHDBSCAN*

In [ ]:

graphhdbscan_rows = []

for graph_method in GRAPH_METHODS:
    print(f"\nGraphHDBSCAN*: {graph_method}")

    raw_graph = SIMILARITY_GRAPHS[graph_method]

    model = GraphCoreSGHDBSCAN(
        min_samples=MIN_SAMPLES_VALUES,
        sim_graph_method="precomputed",
        n_neighbors=K_NEIGHBORS,
        min_cluster_size=MIN_CLUSTER_SIZE,
        add_neighbor=ADD_NEIGHBOR,
        no_noise=False,
        use_sparse_fit=True,
    )

    start = time.perf_counter()
    model.fit(raw_graph)
    runtime = time.perf_counter() - start

    tree_store = getattr(model, "condensed_trees_", None)

    if not tree_store:
        tree_store = getattr(
            getattr(model, "coresg_", None),
            "condensed_trees_",
            None,
        )

    if not tree_store:
        raise RuntimeError(
            f"No GraphHDBSCAN* condensed trees for {graph_method}."
        )

    initial_components = nx.number_connected_components(raw_graph)
    post_wss_components = nx.number_connected_components(
        model.dissimilarity_graph_
    )

    for min_samples in sorted(tree_store):
        graphhdbscan_rows.append(
            {
                "method": "GraphHDBSCAN*",
                "graph_method": graph_method,
                "metric": GRAPH_METRIC,
                "k": K_NEIGHBORS,
                "min_cluster_size": MIN_CLUSTER_SIZE,
                "min_samples": int(min_samples),
                "HAI": condensed_tree_hai(tree_store[min_samples], gt=GT),
                "initial_graph_components": initial_components,
                "post_WSS_components_before_native_bridging": post_wss_components,
                "runtime_seconds": runtime,
            }
        )


graphhdbscan_results = (
    pd.DataFrame(graphhdbscan_rows)
    .sort_values("HAI", ascending=False)
    .reset_index(drop=True)
)

display(
    graphhdbscan_results.style.format(
        {"HAI": "{:.6f}", "runtime_seconds": "{:.2f}"}
    )
)



GraphHDBSCAN*: sc_umap


[TIMER] GraphCoreSGHDBSCAN._coerce_precomputed_graph: 2.6416s


[TIMER] GraphCoreSGHDBSCAN._create_similarity_sparse: 4.3838s


[TIMER] GraphCoreSGHDBSCAN._wss_similarity_sparse: 4.6257s


[TIMER] GraphCoreSGHDBSCAN.similarity_to_dissimilarity_sparse: 2.3746s


...

[CORE-SG] m=47: MST+tree+labels in 2.6571s


[CORE-SG] m=48: MST+tree+labels in 2.6004s


[CORE-SG] m=49: MST+tree+labels in 2.5590s


[CORE-SG] m=50: MST+tree+labels in 2.6142s
[TIMER] fit:coresg_.run: 150.1384s
[TIMER] GraphCoreSGHDBSCAN.fit: 193.4804s


,method,graph_method,metric,k,min_cluster_size,min_samples,HAI,initial_graph_components,post_WSS_components_before_native_bridging,runtime_seconds
0,GraphHDBSCAN*,sc_gauss,euclidean,15,55,15,0.833049,1,1,173.85
1,GraphHDBSCAN*,sc_gauss,euclidean,15,55,14,0.830504,1,1,173.85
2,GraphHDBSCAN*,sc_gauss,euclidean,15,55,21,0.827594,1,1,173.85
3,GraphHDBSCAN*,jaccard_phenograph,euclidean,15,55,25,0.826018,2,2,193.48
4,GraphHDBSCAN*,sc_gauss,euclidean,15,55,10,0.825485,1,1,173.85
5,GraphHDBSCAN*,sc_gauss,euclidean,15,55,20,0.825097,1,1,173.85
6,GraphHDBSCAN*,jaccard_phenograph,euclidean,15,55,26,0.825073,2,2,193.48
7,GraphHDBSCAN*,sc_gauss,euclidean,15,55,9,0.823706,1,1,173.85
8,GraphHDBSCAN*,sc_gauss,euclidean,15,55,13,0.822930,1,1,173.85
9,GraphHDBSCAN*,sc_gauss,euclidean,15,55,18,0.822720,1,1,173.85


## 7. HDBSCAN*

In [ ]:

hdbscan_rows = []

for min_samples in [15, 41]:
    print(f"HDBSCAN*: min_samples={min_samples}")

    model = hdbscan.HDBSCAN(
        min_cluster_size=MIN_CLUSTER_SIZE,
        min_samples=min_samples,
        metric="euclidean",
        algorithm="best",
        approx_min_span_tree=False,
        cluster_selection_method="eom",
        core_dist_n_jobs=-1,
    )

    start = time.perf_counter()
    model.fit(X_dense)
    runtime = time.perf_counter() - start

    hdbscan_rows.append(
        {
            "method": "HDBSCAN*",
            "graph_method": np.nan,
            "metric": "euclidean",
            "k": np.nan,
            "min_cluster_size": MIN_CLUSTER_SIZE,
            "min_samples": min_samples,
            "HAI": condensed_tree_hai(model.condensed_tree_, gt=GT),
            "runtime_seconds": runtime,
        }
    )


hdbscan_results = (
    pd.DataFrame(hdbscan_rows)
    .sort_values("HAI", ascending=False)
    .reset_index(drop=True)
)

display(
    hdbscan_results.style.format(
        {"HAI": "{:.6f}", "runtime_seconds": "{:.2f}"}
    )
)


HDBSCAN*: min_samples=15


HDBSCAN*: min_samples=41


,method,graph_method,metric,k,min_cluster_size,min_samples,HAI,runtime_seconds
0,HDBSCAN*,nan,euclidean,nan,55,41,0.725631,4168.90
1,HDBSCAN*,nan,euclidean,nan,55,15,0.718819,4163.19


## 8. HDBSCAN* + PCA

In [ ]:

pca = PCA(n_components=PCA_COMPONENTS, svd_solver="full")

start = time.perf_counter()
X_pca = pca.fit_transform(X_dense)
pca_runtime = time.perf_counter() - start

X_pca = np.asarray(X_pca, dtype=np.float64, order="C")

pca_rows = []

for min_samples in MIN_SAMPLES_VALUES:
    print(f"HDBSCAN*+PCA: min_samples={min_samples}")

    model = hdbscan.HDBSCAN(
        min_cluster_size=MIN_CLUSTER_SIZE,
        min_samples=min_samples,
        metric="euclidean",
        algorithm="best",
        approx_min_span_tree=False,
        cluster_selection_method="eom",
        core_dist_n_jobs=-1,
    )

    start = time.perf_counter()
    model.fit(X_pca)
    runtime = time.perf_counter() - start

    pca_rows.append(
        {
            "method": "HDBSCAN*+PCA",
            "graph_method": np.nan,
            "metric": "euclidean",
            "k": np.nan,
            "min_cluster_size": MIN_CLUSTER_SIZE,
            "min_samples": min_samples,
            "pca_components": PCA_COMPONENTS,
            "HAI": condensed_tree_hai(model.condensed_tree_, gt=GT),
            "pca_runtime_seconds": pca_runtime,
            "runtime_seconds": runtime,
        }
    )


hdbscan_pca_results = (
    pd.DataFrame(pca_rows)
    .sort_values("HAI", ascending=False)
    .reset_index(drop=True)
)

display(
    hdbscan_pca_results.style.format(
        {
            "HAI": "{:.6f}",
            "pca_runtime_seconds": "{:.2f}",
            "runtime_seconds": "{:.2f}",
        }
    )
)


HDBSCAN*+PCA: min_samples=2


HDBSCAN*+PCA: min_samples=3


HDBSCAN*+PCA: min_samples=4


HDBSCAN*+PCA: min_samples=5


HDBSCAN*+PCA: min_samples=6


...

HDBSCAN*+PCA: min_samples=47


HDBSCAN*+PCA: min_samples=48


HDBSCAN*+PCA: min_samples=49


HDBSCAN*+PCA: min_samples=50


,method,graph_method,metric,k,min_cluster_size,min_samples,pca_components,HAI,pca_runtime_seconds,runtime_seconds
0,HDBSCAN*+PCA,nan,euclidean,nan,55,41,15,0.717903,4.23,4.97
1,HDBSCAN*+PCA,nan,euclidean,nan,55,50,15,0.717269,4.23,4.25
2,HDBSCAN*+PCA,nan,euclidean,nan,55,49,15,0.717204,4.23,4.39
3,HDBSCAN*+PCA,nan,euclidean,nan,55,42,15,0.717075,4.23,4.97
4,HDBSCAN*+PCA,nan,euclidean,nan,55,48,15,0.716914,4.23,4.73
5,HDBSCAN*+PCA,nan,euclidean,nan,55,46,15,0.716276,4.23,4.47
6,HDBSCAN*+PCA,nan,euclidean,nan,55,47,15,0.716142,4.23,4.40
7,HDBSCAN*+PCA,nan,euclidean,nan,55,43,15,0.715796,4.23,4.40
8,HDBSCAN*+PCA,nan,euclidean,nan,55,45,15,0.715539,4.23,4.61
9,HDBSCAN*+PCA,nan,euclidean,nan,55,44,15,0.715174,4.23,4.99


## 9. AHSCAN

In [ ]:

def edge_structural_similarities(graph):
    if graph.is_directed():
        raise ValueError("AHSCAN requires an undirected graph.")
    if graph.is_multigraph():
        raise ValueError("AHSCAN requires a simple graph.")
    if nx.number_of_selfloops(graph):
        raise ValueError("Self-loops are not supported.")

    nodes = sorted(graph.nodes())
    rank = {node: index for index, node in enumerate(nodes)}

    neighborhoods = {
        node: set(graph.neighbors(node)) | {node}
        for node in nodes
    }

    similarities = {}

    for u, v in graph.edges():
        edge = (u, v) if rank[u] < rank[v] else (v, u)

        gu = neighborhoods[edge[0]]
        gv = neighborhoods[edge[1]]

        similarities[edge] = (
            len(gu.intersection(gv))
            / math.sqrt(len(gu) * len(gv))
        )

    return similarities, rank


def ahscan_exact_hai_forest(graph, gt=GT, progress_every=5000):
    """
    Exact AHSCAN HAI for a possibly disconnected initial graph.
    """
    start = time.perf_counter()

    graph = nx.Graph(graph)
    n = len(gt["encoded"])
    graph.add_nodes_from(range(n))

    if set(graph.nodes()) != set(range(n)):
        raise ValueError("Graph nodes must be exactly 0,...,n-1.")

    components = [set(c) for c in nx.connected_components(graph)]

    similarities, rank = edge_structural_similarities(graph)

    ordered_edges = sorted(
        similarities,
        key=lambda edge: (
            -similarities[edge],
            rank[edge[0]],
            rank[edge[1]],
        ),
    )

    n_labels = len(gt["label_order"])

    parent = np.arange(n, dtype=np.int64)
    sizes = np.ones(n, dtype=np.int64)

    histograms = np.zeros((n, n_labels), dtype=np.int64)
    histograms[np.arange(n), gt["encoded"]] = 1

    def find(node):
        node = int(node)

        while parent[node] != node:
            parent[node] = parent[parent[node]]
            node = int(parent[node])

        return node

    absolute_difference = 0.0
    counted_pairs = 0
    merge_count = 0

    for u, v in ordered_edges:
        ru = find(u)
        rv = find(v)

        if ru == rv:
            continue

        merged_size = int(sizes[ru] + sizes[rv])

        pair_counts = (
            np.outer(histograms[ru], histograms[rv])
            + np.outer(histograms[rv], histograms[ru])
        )

        predicted_distance = merged_size / n

        absolute_difference += np.sum(
            pair_counts * np.abs(predicted_distance - gt["distance"]),
            dtype=np.float64,
        )

        counted_pairs += int(pair_counts.sum())

        if rank[ru] > rank[rv]:
            ru, rv = rv, ru

        parent[rv] = ru
        sizes[ru] += sizes[rv]
        histograms[ru] += histograms[rv]

        merge_count += 1

        if progress_every and merge_count % progress_every == 0:
            print(f"AHSCAN merges: {merge_count:,}")

    root_error, root_pairs = forest_root_contribution(components, gt=gt)

    absolute_difference += root_error
    counted_pairs += root_pairs

    expected_merges = n - len(components)
    expected_pairs = n * (n - 1)

    if merge_count != expected_merges:
        raise ValueError(
            f"AHSCAN produced {merge_count} merges; expected {expected_merges}."
        )

    if counted_pairs != expected_pairs:
        raise ValueError(
            f"AHSCAN counted {counted_pairs} ordered pairs; "
            f"expected {expected_pairs}."
        )

    return {
        "HAI": float(1.0 - absolute_difference / (n * n)),
        "components": len(components),
        "merge_count": merge_count,
        "runtime_seconds": float(time.perf_counter() - start),
    }


## 10. DHSCAN

In [ ]:

def dhscan_exact_hai_forest(
    graph,
    gt=GT,
    progress_every_edges=10000,
    progress_every_splits=5000,
):
    """
    Exact DHSCAN HAI for a possibly disconnected initial graph.
    """
    start = time.perf_counter()

    graph = nx.Graph(graph)
    n = len(gt["encoded"])
    graph.add_nodes_from(range(n))

    if set(graph.nodes()) != set(range(n)):
        raise ValueError("Graph nodes must be exactly 0,...,n-1.")

    initial_components = [set(c) for c in nx.connected_components(graph)]

    absolute_difference = 0.0
    counted_pairs = 0
    total_removed_edges = 0
    total_splits = 0

    root_error, root_pairs = forest_root_contribution(
        initial_components,
        gt=gt,
    )

    absolute_difference += root_error
    counted_pairs += root_pairs

    for component_index, component in enumerate(initial_components, start=1):
        nodes = sorted(component)
        m = len(nodes)

        if m <= 1:
            continue

        work = graph.subgraph(nodes).copy()
        rank = {node: i for i, node in enumerate(nodes)}

        def canonical_edge(u, v):
            return (u, v) if rank[u] < rank[v] else (v, u)

        gamma = {
            node: set(work.neighbors(node)) | {node}
            for node in nodes
        }

        version = {node: 0 for node in nodes}
        priority_queue = []

        def push_edge(u, v):
            a, b = canonical_edge(u, v)

            similarity = (
                len(gamma[a].intersection(gamma[b]))
                / math.sqrt(len(gamma[a]) * len(gamma[b]))
            )

            heapq.heappush(
                priority_queue,
                (
                    similarity,
                    rank[a],
                    rank[b],
                    a,
                    b,
                    version[a],
                    version[b],
                ),
            )

        for u, v in work.edges():
            push_edge(u, v)

        active_nodes = {0: set(nodes)}
        active_histograms = {0: reference_histogram(nodes, gt=gt)}
        node_component = {node: 0 for node in nodes}
        next_component_id = 1

        component_splits = 0
        component_counted_pairs = 0

        while work.number_of_edges() > 0:
            while priority_queue:
                (
                    _similarity,
                    _,
                    _,
                    u,
                    v,
                    version_u,
                    version_v,
                ) = heapq.heappop(priority_queue)

                if not work.has_edge(u, v):
                    continue

                if (
                    version[u] != version_u
                    or version[v] != version_v
                ):
                    continue

                break
            else:
                raise RuntimeError(
                    "DHSCAN priority queue lost all active edges."
                )

            work.remove_edge(u, v)

            total_removed_edges += 1

            gamma[u].remove(v)
            gamma[v].remove(u)

            version[u] += 1
            version[v] += 1

            for neighbor in list(work.neighbors(u)):
                push_edge(u, neighbor)

            for neighbor in list(work.neighbors(v)):
                push_edge(v, neighbor)

            if (
                progress_every_edges
                and total_removed_edges % progress_every_edges == 0
            ):
                print(f"DHSCAN removed edges: {total_removed_edges:,}")

            if nx.has_path(work, u, v):
                continue

            parent_id = node_component[u]

            if parent_id != node_component[v]:
                raise RuntimeError(
                    "DHSCAN split endpoints were already in different nodes."
                )

            parent_nodes = active_nodes[parent_id]
            parent_histogram = active_histograms[parent_id]

            component_u = set(nx.node_connected_component(work, u))
            component_v = parent_nodes - component_u

            if v not in component_v:
                raise RuntimeError("DHSCAN split reconstruction failed.")

            nodes_u = np.fromiter(
                component_u,
                dtype=np.int64,
                count=len(component_u),
            )

            histogram_u = np.bincount(
                gt["encoded"][nodes_u],
                minlength=len(gt["label_order"]),
            ).astype(np.int64)

            histogram_v = parent_histogram - histogram_u

            parent_size = len(parent_nodes)

            pair_counts = (
                np.outer(histogram_u, histogram_v)
                + np.outer(histogram_v, histogram_u)
            )

            predicted_distance = parent_size / n

            absolute_difference += np.sum(
                pair_counts * np.abs(predicted_distance - gt["distance"]),
                dtype=np.float64,
            )

            pairs_here = int(pair_counts.sum())

            counted_pairs += pairs_here
            component_counted_pairs += pairs_here

            if len(component_u) >= len(component_v):
                keep_nodes = component_u
                keep_hist = histogram_u
                new_nodes = component_v
                new_hist = histogram_v
            else:
                keep_nodes = component_v
                keep_hist = histogram_v
                new_nodes = component_u
                new_hist = histogram_u

            active_nodes[parent_id] = keep_nodes
            active_histograms[parent_id] = keep_hist

            new_id = next_component_id
            next_component_id += 1

            active_nodes[new_id] = new_nodes
            active_histograms[new_id] = new_hist

            for node in new_nodes:
                node_component[node] = new_id

            component_splits += 1
            total_splits += 1

            if (
                progress_every_splits
                and total_splits % progress_every_splits == 0
            ):
                print(f"DHSCAN splits: {total_splits:,}")

        if component_splits != m - 1:
            raise ValueError(
                f"DHSCAN component {component_index}: "
                f"{component_splits} splits, expected {m - 1}."
            )

        if component_counted_pairs != m * (m - 1):
            raise ValueError(
                f"DHSCAN component {component_index}: "
                f"{component_counted_pairs} ordered pairs, "
                f"expected {m * (m - 1)}."
            )

    expected_splits = n - len(initial_components)
    expected_pairs = n * (n - 1)

    if total_splits != expected_splits:
        raise ValueError(
            f"DHSCAN produced {total_splits} splits; "
            f"expected {expected_splits}."
        )

    if counted_pairs != expected_pairs:
        raise ValueError(
            f"DHSCAN counted {counted_pairs} ordered pairs; "
            f"expected {expected_pairs}."
        )

    return {
        "HAI": float(1.0 - absolute_difference / (n * n)),
        "components": len(initial_components),
        "split_count": total_splits,
        "removed_edges": total_removed_edges,
        "runtime_seconds": float(time.perf_counter() - start),
    }


## 11. Paris

In [ ]:


from scipy import sparse


# ------------------------------------------------------------
# 1. Corrected Paris component function
# ------------------------------------------------------------

def paris_component_contribution(
    weighted_graph,
    component_nodes,
    gt=GT,
):
    global_n = len(gt["encoded"])

    nodes = sorted(component_nodes)
    m = len(nodes)

    if m <= 1:
        return 0.0, 0, 0

    # --------------------------------------------------------
    # Convert NetworkX graph -> SciPy sparse matrix
    # --------------------------------------------------------

    adjacency = nx.to_scipy_sparse_array(
        weighted_graph,
        nodelist=nodes,
        weight="weight",
        format="csr",
        dtype=np.float64,
    )

    # scikit-network 0.33.5 requires csr_matrix,
    # not the newer csr_array.
    adjacency = sparse.csr_matrix(
        adjacency,
        dtype=np.float64,
    )

    adjacency.sum_duplicates()
    adjacency.eliminate_zeros()
    adjacency.sort_indices()

    # --------------------------------------------------------
    # CRITICAL FIX
    #
    # On newer Python / SciPy / NetworkX versions,
    # CSR indices may be int64.
    #
    # scikit-network's compiled Paris code expects
    # C int == np.int32.
    # --------------------------------------------------------

    adjacency.indices = np.asarray(
        adjacency.indices,
        dtype=np.int32,
    )

    adjacency.indptr = np.asarray(
        adjacency.indptr,
        dtype=np.int32,
    )

    adjacency.data = np.asarray(
        adjacency.data,
        dtype=np.float64,
    )

    # Make sure the result is still explicitly CSR.
    adjacency = sparse.csr_matrix(
        (
            adjacency.data,
            adjacency.indices,
            adjacency.indptr,
        ),
        shape=adjacency.shape,
    )

    # Diagnostics
    print(
        f"      Paris matrix: "
        f"{adjacency.shape}, "
        f"nnz={adjacency.nnz:,}, "
        f"indices={adjacency.indices.dtype}, "
        f"indptr={adjacency.indptr.dtype}, "
        f"data={adjacency.data.dtype}"
    )

    if adjacency.indices.dtype != np.int32:
        raise TypeError(
            "Paris CSR indices are not int32."
        )

    if adjacency.indptr.dtype != np.int32:
        raise TypeError(
            "Paris CSR indptr is not int32."
        )

    # --------------------------------------------------------
    # Run Paris
    # --------------------------------------------------------

    model = Paris(
        weights="degree",
        reorder=False,
    )

    dendrogram = np.asarray(
        model.fit_predict(adjacency),
        dtype=np.float64,
    )

    if dendrogram.shape != (m - 1, 4):
        raise ValueError(
            f"Paris dendrogram shape "
            f"{dendrogram.shape}; "
            f"expected {(m - 1, 4)}."
        )

    # --------------------------------------------------------
    # Exact HAI contribution
    # --------------------------------------------------------

    local_encoded = gt["encoded"][
        np.asarray(
            nodes,
            dtype=np.int64,
        )
    ]

    n_labels = len(
        gt["label_order"]
    )

    histograms = np.zeros(
        (2 * m - 1, n_labels),
        dtype=np.int64,
    )

    sizes = np.zeros(
        2 * m - 1,
        dtype=np.int64,
    )

    histograms[
        np.arange(m),
        local_encoded,
    ] = 1

    sizes[:m] = 1

    absolute_difference = 0.0
    counted_pairs = 0

    for row_index, row in enumerate(
        dendrogram
    ):
        left = int(row[0])
        right = int(row[1])

        # With reorder=False,
        # row k creates node m+k.
        node = m + row_index

        if (
            left >= node
            or right >= node
        ):
            raise ValueError(
                "Paris returned invalid child IDs "
                "despite reorder=False."
            )

        merged_size = int(
            sizes[left]
            + sizes[right]
        )

        reported_size = int(
            round(
                float(row[3])
            )
        )

        if reported_size != merged_size:
            raise ValueError(
                f"Paris size mismatch "
                f"at local node {node}: "
                f"{reported_size} "
                f"!= {merged_size}"
            )

        pair_counts = (
            np.outer(
                histograms[left],
                histograms[right],
            )
            +
            np.outer(
                histograms[right],
                histograms[left],
            )
        )

        # IMPORTANT:
        # HAI hierarchy size is normalized by the
        # complete dataset size N, not component size m.
        predicted_distance = (
            merged_size
            / global_n
        )

        absolute_difference += np.sum(
            pair_counts
            * np.abs(
                predicted_distance
                - gt["distance"]
            ),
            dtype=np.float64,
        )

        counted_pairs += int(
            pair_counts.sum()
        )

        histograms[node] = (
            histograms[left]
            + histograms[right]
        )

        sizes[node] = (
            merged_size
        )

    expected_pairs = (
        m * (m - 1)
    )

    if counted_pairs != expected_pairs:
        raise ValueError(
            f"Paris component counted "
            f"{counted_pairs} ordered pairs; "
            f"expected {expected_pairs}."
        )

    return (
        float(
            absolute_difference
        ),
        int(
            counted_pairs
        ),
        m - 1,
    )


# ------------------------------------------------------------
# 2. Paris HAI for potentially disconnected graph
# ------------------------------------------------------------

def paris_exact_hai_forest(
    weighted_graph,
    gt=GT,
):
    start = time.perf_counter()

    graph = nx.Graph(
        weighted_graph
    )

    n = len(
        gt["encoded"]
    )

    graph.add_nodes_from(
        range(n)
    )

    if set(graph.nodes()) != set(range(n)):
        raise ValueError(
            "Graph nodes must be "
            "exactly 0,...,n-1."
        )

    components = [
        set(component)
        for component
        in nx.connected_components(
            graph
        )
    ]

    absolute_difference = 0.0
    counted_pairs = 0
    merge_count = 0

    print(
        f"    Paris: "
        f"{len(components)} "
        f"connected component(s)"
    )

    for component_index, component in enumerate(
        components,
        start=1,
    ):
        print(
            f"    component "
            f"{component_index}/"
            f"{len(components)} "
            f"(n={len(component):,})"
        )

        (
            error,
            pairs,
            merges,
        ) = paris_component_contribution(
            graph,
            component,
            gt=gt,
        )

        absolute_difference += (
            error
        )

        counted_pairs += (
            pairs
        )

        merge_count += (
            merges
        )

    # --------------------------------------------------------
    # Different original graph components meet only at
    # the synthetic hierarchy root.
    # --------------------------------------------------------

    (
        root_error,
        root_pairs,
    ) = forest_root_contribution(
        components,
        gt=gt,
    )

    absolute_difference += (
        root_error
    )

    counted_pairs += (
        root_pairs
    )

    expected_pairs = (
        n * (n - 1)
    )

    expected_merges = (
        n - len(components)
    )

    if counted_pairs != expected_pairs:
        raise ValueError(
            f"Paris counted "
            f"{counted_pairs} "
            f"ordered pairs; "
            f"expected {expected_pairs}."
        )

    if merge_count != expected_merges:
        raise ValueError(
            f"Paris produced "
            f"{merge_count} "
            f"within-component merges; "
            f"expected {expected_merges}."
        )

    hai = float(
        1.0
        - absolute_difference
        / (n * n)
    )

    return {
        "HAI": hai,
        "components": int(
            len(components)
        ),
        "merge_count": int(
            merge_count
        ),
        "runtime_seconds": float(
            time.perf_counter()
            - start
        ),
    }




## 12. Run AHSCAN, DHSCAN, Paris

In [ ]:

external_rows = []

for graph_method in GRAPH_METHODS:
    weighted_graph = SIMILARITY_GRAPHS[graph_method]

    topology_graph = nx.Graph()
    topology_graph.add_nodes_from(range(N))
    topology_graph.add_edges_from(weighted_graph.edges())

    print(
        f"\n{graph_method}: "
        f"{nx.number_connected_components(topology_graph)} "
        "initial component(s)"
    )

    if RUN_AHSCAN:
        print("  AHSCAN")
        result = ahscan_exact_hai_forest(topology_graph, gt=GT)

        external_rows.append(
            {
                "method": "AHSCAN",
                "graph_method": graph_method,
                "metric": GRAPH_METRIC,
                "k": K_NEIGHBORS,
                "min_cluster_size": np.nan,
                "min_samples": np.nan,
                "HAI": result["HAI"],
                "initial_graph_components": result["components"],
                "runtime_seconds": result["runtime_seconds"],
            }
        )

    if RUN_DHSCAN:
        print("  DHSCAN")
        result = dhscan_exact_hai_forest(topology_graph, gt=GT)

        external_rows.append(
            {
                "method": "DHSCAN",
                "graph_method": graph_method,
                "metric": GRAPH_METRIC,
                "k": K_NEIGHBORS,
                "min_cluster_size": np.nan,
                "min_samples": np.nan,
                "HAI": result["HAI"],
                "initial_graph_components": result["components"],
                "runtime_seconds": result["runtime_seconds"],
            }
        )

    if RUN_PARIS:
        print("  Paris")
        result = paris_exact_hai_forest(weighted_graph, gt=GT)

        external_rows.append(
            {
                "method": "Paris",
                "graph_method": graph_method,
                "metric": GRAPH_METRIC,
                "k": K_NEIGHBORS,
                "min_cluster_size": np.nan,
                "min_samples": np.nan,
                "HAI": result["HAI"],
                "initial_graph_components": result["components"],
                "runtime_seconds": result["runtime_seconds"],
            }
        )


external_graph_results = (
    pd.DataFrame(external_rows)
    .sort_values("HAI", ascending=False)
    .reset_index(drop=True)
)

display(
    external_graph_results.style.format(
        {"HAI": "{:.6f}", "runtime_seconds": "{:.2f}"}
    )
)



sc_umap: 1 initial component(s)
  AHSCAN


AHSCAN merges: 5,000
AHSCAN merges: 10,000


AHSCAN merges: 15,000
AHSCAN merges: 20,000


AHSCAN merges: 25,000
AHSCAN merges: 30,000


AHSCAN merges: 35,000
AHSCAN merges: 40,000


...

  Paris


    Paris: 2 connected component(s)
    component 1/2 (n=71,215)


      Paris matrix: (71215, 71215), nnz=1,597,710, indices=int32, indptr=int32, data=float64


    component 2/2 (n=45)
      Paris matrix: (45, 45), nnz=850, indices=int32, indptr=int32, data=float64


,method,graph_method,metric,k,min_cluster_size,min_samples,HAI,initial_graph_components,runtime_seconds
0,Paris,jaccard_phenograph,euclidean,15,nan,nan,0.752018,2,14.35
1,Paris,sc_umap,euclidean,15,nan,nan,0.751758,1,17.86
2,Paris,sc_gauss,euclidean,15,nan,nan,0.749031,1,16.64
3,DHSCAN,sc_gauss,euclidean,15,nan,nan,0.664143,1,25233.58
4,DHSCAN,sc_umap,euclidean,15,nan,nan,0.664143,1,24418.43
5,DHSCAN,jaccard_phenograph,euclidean,15,nan,nan,0.654598,2,22442.52
6,AHSCAN,sc_umap,euclidean,15,nan,nan,0.636972,1,11.25
7,AHSCAN,sc_gauss,euclidean,15,nan,nan,0.636972,1,10.32
8,AHSCAN,jaccard_phenograph,euclidean,15,nan,nan,0.636365,2,10.33


## 13. Final result tables

In [ ]:

frames = [
    graphhdbscan_results,
    hdbscan_results,
    hdbscan_pca_results,
]

if not external_graph_results.empty:
    frames.append(external_graph_results)

all_results = pd.concat(frames, ignore_index=True, sort=False)

method_order = {
    "GraphHDBSCAN*": 0,
    "HDBSCAN*": 1,
    "HDBSCAN*+PCA": 2,
    "AHSCAN": 3,
    "DHSCAN": 4,
    "Paris": 5,
}

all_results["_order"] = all_results["method"].map(method_order)

all_results = (
    all_results
    .sort_values(["_order", "HAI"], ascending=[True, False])
    .drop(columns="_order")
    .reset_index(drop=True)
)

print("ALL TESTED CONFIGURATIONS")
display(
    all_results.style.format(
        {
            "HAI": "{:.6f}",
            "runtime_seconds": "{:.2f}",
            "pca_runtime_seconds": "{:.2f}",
        },
        na_rep="",
    )
)


best_indices = (
    all_results
    .groupby("method")["HAI"]
    .idxmax()
)

best_by_method = all_results.loc[best_indices].copy()
best_by_method["_order"] = best_by_method["method"].map(method_order)

best_by_method = (
    best_by_method
    .sort_values("_order")
    .drop(columns="_order")
    .reset_index(drop=True)
)

print("\nBEST RESULT FOR EACH METHOD")

best_columns = [
    "method",
    "HAI",
    "graph_method",
    "metric",
    "k",
    "min_cluster_size",
    "min_samples",
]

display(
    best_by_method[
        [c for c in best_columns if c in best_by_method.columns]
    ].style.format(
        {"HAI": "{:.6f}"},
        na_rep="",
    )
)


graph_based = all_results[
    all_results["graph_method"].notna()
].copy()

best_method_graph_indices = (
    graph_based
    .groupby(["method", "graph_method"])["HAI"]
    .idxmax()
)

best_by_method_and_graph = (
    graph_based
    .loc[best_method_graph_indices]
    .sort_values(["method", "graph_method"])
    .reset_index(drop=True)
)

print("\nBEST RESULT FOR EACH METHOD × GRAPH")

display(
    best_by_method_and_graph[
        [c for c in best_columns if c in best_by_method_and_graph.columns]
    ].style.format(
        {"HAI": "{:.6f}"},
        na_rep="",
    )
)


outputs = {
    "bone_marrow_HAI_k15_all_configurations.csv": all_results,
    "bone_marrow_HAI_k15_best_each_method.csv": best_by_method,
    "bone_marrow_HAI_k15_best_each_method_graph.csv": best_by_method_and_graph,
    "bone_marrow_HAI_k15_graph_build_info.csv": graph_build_info,
    "bone_marrow_HAI_reference_direct_assignments.csv": GT["direct_assignments"],
}

for filename, frame in outputs.items():
    frame.to_csv(filename, index=False)
    print("Saved:", Path(filename).resolve())


ALL TESTED CONFIGURATIONS


,method,graph_method,metric,k,min_cluster_size,min_samples,HAI,initial_graph_components,post_WSS_components_before_native_bridging,runtime_seconds,pca_components,pca_runtime_seconds
0,GraphHDBSCAN*,sc_gauss,euclidean,15.000000,55.000000,15.000000,0.833049,1.000000,1.000000,173.85,,
1,GraphHDBSCAN*,sc_gauss,euclidean,15.000000,55.000000,14.000000,0.830504,1.000000,1.000000,173.85,,
2,GraphHDBSCAN*,sc_gauss,euclidean,15.000000,55.000000,21.000000,0.827594,1.000000,1.000000,173.85,,
3,GraphHDBSCAN*,jaccard_phenograph,euclidean,15.000000,55.000000,25.000000,0.826018,2.000000,2.000000,193.48,,
4,GraphHDBSCAN*,sc_gauss,euclidean,15.000000,55.000000,10.000000,0.825485,1.000000,1.000000,173.85,,
5,GraphHDBSCAN*,sc_gauss,euclidean,15.000000,55.000000,20.000000,0.825097,1.000000,1.000000,173.85,,
6,GraphHDBSCAN*,jaccard_phenograph,euclidean,15.000000,55.000000,26.000000,0.825073,2.000000,2.000000,193.48,,
7,GraphHDBSCAN*,sc_gauss,euclidean,15.000000,55.000000,9.000000,0.823706,1.000000,1.000000,173.85,,
8,GraphHDBSCAN*,sc_gauss,euclidean,15.000000,55.000000,13.000000,0.822930,1.000000,1.000000,173.85,,
9,GraphHDBSCAN*,sc_gauss,euclidean,15.000000,55.000000,18.000000,0.822720,1.000000,1.000000,173.85,,



BEST RESULT FOR EACH METHOD


,method,HAI,graph_method,metric,k,min_cluster_size,min_samples
0,GraphHDBSCAN*,0.833049,sc_gauss,euclidean,15.000000,55.000000,15.000000
1,HDBSCAN*,0.725631,,euclidean,,55.000000,41.000000
2,HDBSCAN*+PCA,0.717903,,euclidean,,55.000000,41.000000
3,AHSCAN,0.636972,sc_umap,euclidean,15.000000,,
4,DHSCAN,0.664143,sc_gauss,euclidean,15.000000,,
5,Paris,0.752018,jaccard_phenograph,euclidean,15.000000,,



BEST RESULT FOR EACH METHOD × GRAPH


,method,HAI,graph_method,metric,k,min_cluster_size,min_samples
0,AHSCAN,0.636365,jaccard_phenograph,euclidean,15.000000,,
1,AHSCAN,0.636972,sc_gauss,euclidean,15.000000,,
2,AHSCAN,0.636972,sc_umap,euclidean,15.000000,,
3,DHSCAN,0.654598,jaccard_phenograph,euclidean,15.000000,,
4,DHSCAN,0.664143,sc_gauss,euclidean,15.000000,,
5,DHSCAN,0.664143,sc_umap,euclidean,15.000000,,
6,GraphHDBSCAN*,0.826018,jaccard_phenograph,euclidean,15.000000,55.000000,25.000000
7,GraphHDBSCAN*,0.833049,sc_gauss,euclidean,15.000000,55.000000,15.000000
8,GraphHDBSCAN*,0.805904,sc_umap,euclidean,15.000000,55.000000,16.000000
9,Paris,0.752018,jaccard_phenograph,euclidean,15.000000,,


Saved: /work/bone_marrow_HAI_k15_all_configurations.csv
Saved: /work/bone_marrow_HAI_k15_best_each_method.csv
Saved: /work/bone_marrow_HAI_k15_best_each_method_graph.csv
Saved: /work/bone_marrow_HAI_k15_graph_build_info.csv
Saved: /work/bone_marrow_HAI_reference_direct_assignments.csv



## Interpretation

`best_by_method` is a best-case/oracle summary because HAI itself is used to choose
the best GraphHDBSCAN* graph/minPts combination and the best `min_samples` value for
HDBSCAN* and HDBSCAN*+PCA.

The disconnected-graph handling does not insert artificial edges into the initial
graph for AHSCAN, DHSCAN, or Paris. Cross-component pairs meet only at a synthetic
top-level root and therefore receive inferred hierarchy distance 1.

For GraphHDBSCAN*, the raw similarity graph is passed unchanged to the package, so
the package performs WSS first and only then connects the dissimilarity graph if
needed.
